# Graph Attention Network #

## Connectivity Matrix ##

In [ ]:
import numpy as np
import torch
from enigmatoolbox.datasets import load_fc, load_sc

def apply_threshold(matrix, top_pct):
    # Keep only the top % of connections (ignoring zeros/diagonal)
    threshold_val = np.percentile(matrix[matrix > 0], 100 - top_pct)
    return np.where(matrix >= threshold_val, matrix, 0)

def get_edge_index(adj_matrix):
    sources, targets = np.nonzero(adj_matrix)
    edge_weights = adj_matrix[sources, targets]
    
    edge_index = torch.tensor(np.vstack((sources, targets)), dtype=torch.long)
    edge_attr = torch.tensor(edge_weights, dtype=torch.float)
    return edge_index, edge_attr

def load_connectivity_400(conn_type='FC', top_pct=10.0):
    """
    Fetches the HCP connectome using the ENIGMA Toolbox.
    """
    if conn_type == 'FC':
        # (cortical_fc, region_labels, subcortical_fc, subcortical_labels)
        matrix, _, _, _ = load_fc('schaefer_400')
    elif conn_type == 'SC':
        # structural connectivity
        matrix, _, _, _ = load_sc('schaefer_400')
    else:
        raise ValueError("conn_type must be 'FC' or 'SC'")
        
    adj = apply_threshold(matrix, top_pct)
    return get_edge_index(adj)

# Test fetching the Functional Connectivity 
edge_index, edge_attr = load_connectivity_400(conn_type='FC', top_pct=10.0)
print(f"Edge Index Shape: {edge_index.shape}")
print(f"Edge Attr Shape: {edge_attr.shape}")

Edge Index Shape: torch.Size([2, 30976])
Edge Attr Shape: torch.Size([30976])


## Node Features
We define the static node features for the 400 regions: their 3D Region Centroids (X, Y, Z) and a One-Hot encoded vector of their Yeo-17 functional network assignment. When training the GAN, the active fMRI beta signal for that specific trial will be concatenated to this vector.

In [25]:
from nilearn import datasets
from nilearn.plotting import find_parcellation_cut_coords

def get_static_node_features(n_rois=400, networks=17):
    """
    Builds the fundamental structure of the Graph Nodes.
    Outputs a tensor of shape (400, 20). 
    Features: 3 (XYZ centroids) + 17 (Network One-Hot Encoding)
    """
    # Fetch atlas and region coordinates
    atlas = datasets.fetch_atlas_schaefer_2018(n_rois=n_rois, yeo_networks=networks)
    centroids = find_parcellation_cut_coords(atlas.maps) # shape: (400, 3)
    
    # Parse network names from labels, skipping the 'Background' label at index 0
    net_names = [label.split('_')[2] for label in atlas.labels[1:]]
    unique_nets = sorted(list(set(net_names)))
    
    # Create One-Hot encoding block
    one_hot = np.zeros((n_rois, networks))
    for i, net in enumerate(net_names):
        one_hot[i, unique_nets.index(net)] = 1.0
        
    # Concatenate [XYZ, One-Hot]
    static_features = np.hstack((centroids, one_hot))
    
    return torch.tensor(static_features, dtype=torch.float), unique_nets

# Test the extraction
static_node_features, network_names = get_static_node_features()
print(f"Static Node Features Shape: {static_node_features.shape}")
print(static_node_features)
print(f"Networks Found ({len(network_names)}): \n{network_names}")

[fetch_atlas_schaefer_2018] Dataset found in /Users/mario/nilearn_data/schaefer_2018
Static Node Features Shape: torch.Size([400, 20])
tensor([[-35.5084, -62.1367, -17.0257,  ...,   0.0000,   1.0000,   0.0000],
        [-23.3766, -72.7294, -10.3540,  ...,   0.0000,   1.0000,   0.0000],
        [-36.4227, -81.2596, -15.8868,  ...,   0.0000,   1.0000,   0.0000],
        ...,
        [ 64.4541, -34.2970,  10.7501,  ...,   1.0000,   0.0000,   0.0000],
        [ 54.8616, -45.6869,  19.5343,  ...,   1.0000,   0.0000,   0.0000],
        [ 61.3651, -39.6356,  21.9868,  ...,   1.0000,   0.0000,   0.0000]])
Networks Found (17): 
['ContA', 'ContB', 'ContC', 'DefaultA', 'DefaultB', 'DefaultC', 'DorsAttnA', 'DorsAttnB', 'LimbicA', 'LimbicB', 'SalVentAttnA', 'SalVentAttnB', 'SomMotA', 'SomMotB', 'TempPar', 'VisCent', 'VisPeri']


## PyTorch Geometric Dataset
We wrap the processed `.npz` files into a standard PyTorch Geometric InMemoryDataset. This automatically concatenates the BOLD betas with our static brain region features, applies the functional connectivity matrix edges to all samples, and encodes the semantic targets.

In [32]:
import os
import json
import torch
import numpy as np
from torch_geometric.data import Data, InMemoryDataset
from torch.utils.data import ConcatDataset

class BOLD5000GraphDataset(InMemoryDataset):
    def __init__(self, root, edge_index, edge_attr, static_node_features, subject='CSI1', transform=None):
        self.edge_index = edge_index
        self.edge_attr = edge_attr
        self.static_features = static_node_features
        self.subject = subject
        self.data_path = os.path.join(root, subject, f"{subject}_schaefer400.npz")
        
        with open('SemanticLabels.json', 'r') as f:
            self.ontology = json.load(f)
        
        self.category_to_id = {cat: i for i, cat in enumerate(self.ontology.keys())}
        
        self.label_to_category = {}
        for category, details in self.ontology.items():
            for label in details['labels']:
                self.label_to_category[label.lower()] = category
                
        super().__init__(root, transform)
        
        # In PyTorch 2.6+, weights_only=True is the default. We need weights_only=False 
        # to load the complex PyTorch Geometric Data objects correctly.
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def processed_file_names(self):
        return [f'{self.subject}_graph_data.pt']

    def process(self):
        print(f"Processing {self.subject} BOLD fMRI into brain graphs...")
        archive = np.load(self.data_path, allow_pickle=True)
        betas = archive['betas']
        labels = archive['labels']
        
        # Extract metadata fields needed for downstream visual mapping
        imgnames = archive['imgnames']
        sessions = archive['sessions']
        local_idxs = archive['local_idxs']
        
        data_list = []
        skipped = 0
        
        for i in range(len(betas)):
            trial_labels = [l.lower() for l in labels[i]]
            found_categories = set(
                self.label_to_category[l] for l in trial_labels if l in self.label_to_category
            )
            
            # Skip trials with conflicting or missing labels for clean training
            if len(found_categories) != 1:
                skipped += 1
                continue
                
            target_category = list(found_categories)[0]
            y = torch.tensor([self.category_to_id[target_category]], dtype=torch.long)
            
            # Node Features: [Beta (1), Static Features (20)] -> (400, 21)
            trial_beta = torch.tensor(betas[i], dtype=torch.float).unsqueeze(1)
            x = torch.cat([trial_beta, self.static_features], dim=1)
            
            # Inject metadata directly into the Data object for tracking
            graph_data = Data(
                x=x, 
                edge_index=self.edge_index, 
                edge_attr=self.edge_attr, 
                y=y,
                imgname=str(imgnames[i]),
                session=int(sessions[i]),
                local_idx=int(local_idxs[i]),
                subject=self.subject
            )
            data_list.append(graph_data)
            
        print(f"[{self.subject}] Created {len(data_list)} graphs. Skipped {skipped} multi-label/unknown trials.")
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

# Instantiate datasets for all 4 subjects and concatenate them into one massive dataset
subjects = ['CSI1', 'CSI2', 'CSI3', 'CSI4']
all_datasets = []

for subj in subjects:
    ds = BOLD5000GraphDataset(
        root='processed_data', 
        edge_index=edge_index, 
        edge_attr=edge_attr, 
        static_node_features=static_node_features,
        subject=subj
    )
    all_datasets.append(ds)

# Merge perfectly into a single dataset
full_dataset = ConcatDataset(all_datasets)

print(f"\n--- Full Dataset Summary ---")
print(f"Total Graphs: {len(full_dataset)}")
print(f"Num Node Features: {all_datasets[0].num_node_features}")
print(f"Num Edge Features: {all_datasets[0].num_edge_features}")
print(f"Classes: {list(all_datasets[0].category_to_id.keys())}")

# Inspect the very first graph of the combined dataset specifically looking at metadata
first_graph = full_dataset[0]
print("\nExample Data Object Metadata:")
print(f"Image: {first_graph.imgname}")
print(f"Subject: {first_graph.subject}, Session: {first_graph.session}, Local ID: {first_graph.local_idx}")
print(f"Target Label ID: {first_graph.y.item()}")
print(f"Shape: {first_graph}")

Processing...


Processing CSI1 BOLD fMRI into brain graphs...
[CSI1] Created 3945 graphs. Skipped 1309 multi-label/unknown trials.


Done!
Processing...


Processing CSI2 BOLD fMRI into brain graphs...
[CSI2] Created 3945 graphs. Skipped 1309 multi-label/unknown trials.


Done!
Processing...


Processing CSI3 BOLD fMRI into brain graphs...
[CSI3] Created 3945 graphs. Skipped 1309 multi-label/unknown trials.


Done!
Processing...


Processing CSI4 BOLD fMRI into brain graphs...
[CSI4] Created 2307 graphs. Skipped 801 multi-label/unknown trials.


Done!



--- Full Dataset Summary ---
Total Graphs: 14142
Num Node Features: 21
Num Edge Features: 1
Classes: ['person', 'food', 'vehicle', 'location', 'object', 'clothing', 'living thing']

Example Data Object Metadata:
Image: n01930112_19568.JPEG
Subject: CSI1, Session: tensor([1]), Local ID: tensor([0])
Target Label ID: 6
Shape: Data(x=[400, 21], edge_index=[2, 30976], edge_attr=[30976], y=[1], imgname='n01930112_19568.JPEG', session=[1], local_idx=[1], subject='CSI1')


## Graph Attention Network Architecture
We construct the GAT. We configure it to return three things on every forward pass:
1. **The Logits:** The final class prediction for the image.
2. **The Graph Embedding:** The flattened representation *before* the final classifier, useful for downstream similarity/clustering analysis.
3. **The Attention Weights:** The dynamic edge importances for interpretability.

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout
from torch_geometric.nn import GATConv, global_mean_pool

class BOLD5000_GAT(torch.nn.Module):
    def __init__(self, num_node_features, num_classes, hidden_channels, heads=4, dropout=0.5):
        super(BOLD5000_GAT, self).__init__()
        
        self.dropout_rate = dropout
        
        # 1st GAT Layer
        self.conv1 = GATConv(
            in_channels=num_node_features, 
            out_channels=hidden_channels, 
            heads=heads, 
            dropout=dropout,
            concat=True # Output size becomes hidden_channels * heads
        )
        
        # 2nd GAT Layer
        self.conv2 = GATConv(
            in_channels=hidden_channels * heads, 
            out_channels=hidden_channels, 
            heads=heads, 
            dropout=dropout,
            concat=False # Average the heads for the final graph representation
        )
        
        # MLP Layer
        self.classifier = Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch_index, edge_attr=None):
        """
        Forward pass.
        - x: Node feature matrix [num_nodes, num_features]
        - edge_index: Graph connectivity matrix [2, num_edges]
        - batch_index: Tells PyG which nodes belong to which graphs in a batched tensor
        """
    
        #Graph Layer1
        x = self.conv1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x) 
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        #GraphLayer2
        x, (attention_edge_index, attention_weights) = self.conv2(
            x, edge_index, edge_attr=edge_attr, return_attention_weights=True
        )
        x = F.elu(x)
        
        #Condense into graph embedding -> one vector size <hidden_channels> for entire scan
        graph_embedding = global_mean_pool(x, batch_index)
        x_dropped = F.dropout(graph_embedding, p=self.dropout_rate, training=self.training)
        
        #  classification
        logits = self.classifier(x_dropped)
        
        return logits, graph_embedding, (attention_edge_index, attention_weights)

# Instantiate the model to verify architecture
num_features = all_datasets[0].num_node_features
num_classes = len(all_datasets[0].category_to_id.keys())

model = BOLD5000_GAT(
    num_node_features=num_features,
    num_classes=num_classes,
    hidden_channels=64, # hyperparam
    heads=4,            # hyperparam
    dropout=0.5
)

print(model)

BOLD5000_GAT(
  (conv1): GATConv(21, 64, heads=4)
  (conv2): GATConv(256, 64, heads=4)
  (classifier): Linear(in_features=64, out_features=7, bias=True)
)


## Data Splits & Training Loop
We split our dataset into train, validation, and test sets. Then, we set up PyTorch Geometric `DataLoader`s and define our training/evaluation logic, properly unpacking the `logits` from the model's 3-tuple return for calculating Cross-Entropy loss.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split

# Dataset Splits
torch.manual_seed(42)

total_size = len(full_dataset)
train_size = int(0.7 * total_size)  # /70/15/15 Training/Val/Test split
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, persistent_workers=True)

print(f"Data Split| Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Training Setup
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps') # Use Apple Silicon GPU
else:
    device = torch.device('cpu')
    
print(f"Using Device: {device}")
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
lossfunc = nn.CrossEntropyLoss()

from tqdm.auto import tqdm

# Training Logic
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    
    # Wrap loader in tqdm for a progress bar
    progress_bar = tqdm(loader, desc="Training Batches", leave=False)
    for data in progress_bar:
        data = data.to(device)
        optimizer.zero_grad()
        
        #only need logits for calculating loss
        logits, _, _ = model(data.x, data.edge_index, data.batch, data.edge_attr)
        loss = criterion(logits, data.y.view(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * data.num_graphs #CEL gives avg loss, we want total loss
        preds = logits.argmax(dim=1)
        correct += (preds == data.y.view(-1)).sum().item()
        
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# Eval Logic
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    
    progress_bar = tqdm(loader, desc="Eval Batches", leave=False)
    for data in progress_bar:
        data = data.to(device)
        
        logits, _, _ = model(data.x, data.edge_index, data.batch, data.edge_attr)
        loss = criterion(logits, data.y.view(-1))
        
        total_loss += loss.item() * data.num_graphs
        preds = logits.argmax(dim=1)
        correct += (preds == data.y.view(-1)).sum().item()
        
    return total_loss / len(loader.dataset), correct / len(loader.dataset)




Data Split| Train: 9899 | Val: 2121 | Test: 2122
Using Device: mps


/Users/mario/Documents/Programming/4107_BOLD5000/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [39]:
# Training  + Eval Loop
epochs = 50
print("\nStarting Training...")
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, lossfunc)
    val_loss, val_acc = evaluate(model, val_loader, lossfunc)
    
    # Print status cleanly
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} || Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")


Starting Training...


Epoch 001 | Train Loss: 1.5889 | Train Acc: 0.2946 || Val Loss: 1.6476 | Val Acc: 0.2975


KeyboardInterrupt: 